# Síntese da t1 com hls4ml


1. **ReuseFactor**: manual, perfil `min`/`mid`/`max` ou por FPS alvo (como o `SetFolding` do FINN).
2. **FIFOs** (`io_stream`): profundidade padrão, fixa, por camada ou otimizada por cosim.
3. **BRAM**: `BramFactor` (pesos grandes viram porta BRAM externa do IP).
4. Conversão, acurácia (3000 imagens), síntese HLS + Vivado com restrição de clock (`vivado_timing.rpt`).


In [ ]:
import os

import numpy as np
import hls4ml
import hls4ml.report as report
from qkeras.utils import load_qmodel

from cnn_base import prepare_input, set_input_precision
from main import apply_dense_strategy, apply_layer_rf_profile, apply_max_threads, make_output_dir, save_accuracy
from utils import fps_profile_name, get_layer_rf_profiles, load_pickle_data, trace_layers
from vivado_utils import add_vivado_timing

## Configuração

**FIFOs**: em `io_stream` cada saída de camada é um `hls::stream` com
`#pragma HLS STREAM variable=... depth=N` no `myproject.cpp`. O padrão do hls4ml é N = nº de pixels
da saída da camada (ex.: 256 para 16x16). FIFOs rasas viram SRL/LUTRAM; FIFOs fundas viram BRAM,
então a profundidade controla diretamente o uso de BRAM. FIFO rasa demais pode travar o dataflow
(deadlock na cosim) ou baixar o throughput.

**BRAM_FACTOR**: pesos com mais elementos que `BRAM_FACTOR` deixam de ser ROM interna e viram
porta `#pragma HLS INTERFACE bram` do IP, a ser preenchida por fora. A área do IP cai, mas não
inclui mais esses pesos (não é comparável com o FINN).

In [ ]:
BASE_DIR = "t1_4"
MODEL_NAME = "model"

# ---- síntese
STRATEGY = "Resource"        # "Resource" ou "Latency"
DENSE_STRATEGY = "Resource"  # None = igual a STRATEGY. Em Latency a Dense 2048->6 precisa ficar em Resource
IO_TYPE = "io_stream"        # "io_stream" ou "io_parallel"
CLOCK_PERIOD_NS = 10
FPGA_PART = "xc7z020clg400-1" # pynqz1 
BACKEND = "Vitis"

# ---- ReuseFactor
#   "manual"            : REUSE_FACTOR em todas as camadas
#   "min" / "mid" / "max": menor / do meio / maior RF válido de cada camada
#   "fps"               : por camada, o maior RF válido que ainda atinge TARGET_FPS
RF_MODE = "fps"
REUSE_FACTOR = 1
TARGET_FPS = 50000

# ---- FIFOs (só io_stream)
#   None             : padrão do hls4ml (nº de pixels da saída de cada camada)
#   int              : mesma profundidade em todas as FIFOs. Ex.: 64
#   dict             : por FIFO, as demais ficam no padrão. Ex.: {"layer3_out": 32}
#                      (os nomes aparecem na célula "FIFOs" abaixo)
FIFO_DEPTH = None
# True: otimização de FIFOs do hls4ml. Começa todas com PROFILING_FIFO_DEPTH, roda csynth + cosim
# já na conversão e reduz cada FIFO à ocupação máxima medida. Demora; ignora FIFO_DEPTH.
FIFO_OPT = False
PROFILING_FIFO_DEPTH = 100_000

# ---- BRAM
BRAM_FACTOR = None           # None = pesos dentro do IP (padrão). Ex.: 1000

# ---- execução
TEST_SIZE = 3000
MAX_THREADS = 1              # limita threads do Vitis/Vivado (memória). None = sem limite
BUILD = dict(reset=False, csim=True, synth=True, cosim=False, validation=False, export=False, vsynth=True)

In [ ]:
qmodel = load_qmodel(f"models/{BASE_DIR}/{MODEL_NAME}.h5")
qmodel.summary()

X_test_raw, y_test = load_pickle_data("./dataset/sat6_test.pkl")
X_test = np.ascontiguousarray(prepare_input(X_test_raw, qmodel))

## ReuseFactor por camada

`get_layer_rf_profiles` converte o modelo uma vez com RF=1 só para ler `n_in`/`n_out` de cada camada
(`backend.get_layer_mult_size`) e a lista de RFs válidos (`backend.get_valid_reuse_factors`).
No modo `fps`, o orçamento é `f_clk / TARGET_FPS` ciclos por imagem e cada camada recebe o maior RF
válido cujo custo estimado cabe nele.

In [ ]:
if RF_MODE == "manual":
    rf_profile, layer_rf = None, None
else:
    profiles = get_layer_rf_profiles(
        qmodel, BACKEND, FPGA_PART, IO_TYPE, CLOCK_PERIOD_NS, STRATEGY, BASE_DIR,
        target_fps=[TARGET_FPS] if RF_MODE == "fps" else None,
    )
    rf_profile = fps_profile_name(TARGET_FPS) if RF_MODE == "fps" else RF_MODE
    layer_rf = profiles[rf_profile]
    print(f"\nPerfil {rf_profile}: {layer_rf}")

suffix = ""
if FIFO_OPT:
    suffix += "_fifoopt"
elif isinstance(FIFO_DEPTH, int):
    suffix += f"_fifo{FIFO_DEPTH}"
elif FIFO_DEPTH:
    suffix += "_fifocustom"
if BRAM_FACTOR is not None:
    suffix += f"_bf{BRAM_FACTOR}"

OUTPUT_DIR = make_output_dir(
    BASE_DIR, IO_TYPE, REUSE_FACTOR, STRATEGY, CLOCK_PERIOD_NS,
    rf_profile=rf_profile, dense_strategy=DENSE_STRATEGY, suffix=suffix,
)
print("OUTPUT_DIR =", OUTPUT_DIR)

## Configuração hls4ml e conversão

In [ ]:
hls_config = hls4ml.utils.config_from_keras_model(
    qmodel, granularity="name", backend=BACKEND, default_reuse_factor=REUSE_FACTOR,
)
if layer_rf is not None:
    apply_layer_rf_profile(hls_config, layer_rf)

hls_config["LayerName"]["output_dense"]["Precision"]["result"] = "fixed<16,6,RND,SAT>"
set_input_precision(hls_config, qmodel)   # porta de entrada com 8 bits em vez de fixed<16,6>
hls_config["Model"]["Strategy"] = STRATEGY
if DENSE_STRATEGY is not None:
    apply_dense_strategy(qmodel, hls_config, DENSE_STRATEGY)
if BRAM_FACTOR is not None:
    hls_config["Model"]["BramFactor"] = BRAM_FACTOR

tb_data = {}
if FIFO_OPT:
    if IO_TYPE != "io_stream":
        raise ValueError("FIFO_OPT só funciona com io_stream")
    hls_config["Flows"] = ["vitis:fifo_depth_optimization"]
    hls4ml.model.optimizer.get_optimizer("vitis:fifo_depth_optimization").configure(
        profiling_fifo_depth=PROFILING_FIFO_DEPTH
    )
    # a cosim de perfilamento precisa de pelo menos 2 imagens no testbench
    tb_data = {
        "input_data_tb": os.path.join(OUTPUT_DIR, "tb_input.npy"),
        "output_data_tb": os.path.join(OUTPUT_DIR, "tb_output.npy"),
    }
    np.save(tb_data["input_data_tb"], X_test[:2])
    np.save(tb_data["output_data_tb"], qmodel.predict(X_test[:2]))

trace_layers(hls_config)

hls_model = hls4ml.converters.convert_from_keras_model(
    qmodel,
    hls_config=hls_config,
    output_dir=OUTPUT_DIR,
    part=FPGA_PART,
    clock_period=CLOCK_PERIOD_NS,
    io_type=IO_TYPE,
    backend=BACKEND,
    **tb_data,
)

## FIFOs

Aplica `FIFO_DEPTH` (se não for `None` e `FIFO_OPT` estiver desligado) e mostra a profundidade final
de cada FIFO, que vai para o `#pragma HLS STREAM` do `myproject.cpp`.

In [ ]:
fifos = {name: var for name, var in hls_model.output_vars.items()
         if "StreamVariable" in type(var).__name__ and var.pragma}

if FIFO_DEPTH is not None and not FIFO_OPT:
    novas = FIFO_DEPTH if isinstance(FIFO_DEPTH, dict) else {name: FIFO_DEPTH for name in fifos}
    for name, depth in novas.items():
        if name not in fifos:
            print(f"[AVISO] FIFO {name!r} não existe; opções: {list(fifos)}")
            continue
        fifos[name].pragma = (fifos[name].pragma[0], int(depth))

for name, var in fifos.items():
    print(f"{name:20s} depth={var.pragma[1]:>7}  shape={var.shape}")

## Compilação e acurácia (bit-exata, csim em C++)

In [ ]:
hls_model.compile()
save_accuracy(qmodel, hls_model, X_test, y_test, OUTPUT_DIR, TEST_SIZE)

In [ ]:
#hls4ml.utils.plot_model(hls_model, show_shapes=True, show_precision=True, to_file=None)

In [ ]:
#from hls4ml.model.profiling import numerical
#numerical(model=qmodel, hls_model=hls_model, X=X_test[:2000], plot="boxplot")

## Síntese

`add_vivado_timing` acrescenta a restrição de clock e o `report_timing_summary` ao
`vivado_synth.tcl`, gerando o `vivado_timing.rpt` que o `relatorio.py` usa para calcular o FPS real.

In [ ]:
add_vivado_timing(OUTPUT_DIR)
if MAX_THREADS is not None:
    apply_max_threads(OUTPUT_DIR, MAX_THREADS)

hls_model.build(**BUILD)

In [ ]:
report.read_vivado_report(OUTPUT_DIR)